In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Madhurjya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Madhurjya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
import pandas as pd

true_df=pd.read_csv("True.csv")
fake_df=pd.read_csv("Fake.csv")
true_df['label']='REAL'
fake_df['label']='FAKE'

In [4]:
real_sample=true_df.sample(n=2000, random_state=42)
fake_sample=fake_df.sample(n=2000, random_state=42)

reduced_df=pd.concat([real_sample, fake_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(reduced_df['label'].value_counts())
print(reduced_df.head())

label
REAL    2000
FAKE    2000
Name: count, dtype: int64
                                               title  \
0  May's government pushes Brexit bill to avoid '...   
1   Trump’s EPA OKs Pesticide That Causes Brain D...   
2  Man arrested at Trump rally said he wanted to ...   
3  Jared Kushner NEVER Registered To Vote As A “F...   
4  MARTHA STEWART Makes Lewd Gesture Towards Trum...   

                                                text       subject  \
0  LONDON (Reuters) - Brexit minister David Davis...     worldnews   
1  Farmworkers were pulled from fields on Friday ...          News   
2  (Reuters) - A man arrested over the weekend tr...  politicsNews   
3  Meanwhile, as President Trump continues to mee...     left-news   
4  Martha, Martha, Martha You re 75-years old! Ti...     left-news   

                 date label  
0  September 6, 2017   REAL  
1        May 15, 2017  FAKE  
2      June 20, 2016   REAL  
3        Sep 29, 2017  FAKE  
4         May 8, 2017  FAKE  


In [5]:
reduced_df['content']=reduced_df['title']+ " "+reduced_df['text']
reduced_df=reduced_df[['content', 'label']]
reduced_df.head()

,content,label
0,May's government pushes Brexit bill to avoid '...,REAL
1,Trump’s EPA OKs Pesticide That Causes Brain D...,FAKE
2,Man arrested at Trump rally said he wanted to ...,REAL
3,Jared Kushner NEVER Registered To Vote As A “F...,FAKE
4,MARTHA STEWART Makes Lewd Gesture Towards Trum...,FAKE


In [6]:
from sklearn.preprocessing import LabelEncoder

label=LabelEncoder()
y_encoded=label.fit_transform(reduced_df['label'])
print(label.classes_)

['FAKE' 'REAL']


In [7]:
import re
import nltk
nltk.download ('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Madhurjya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
stop_words=set(stopwords.words('english'))
def clean_text(text):
    text=text.lower()
    text=re.sub(r'[^a-z\s]', '', text)
    words=text.split()
    words=[w for w in words if w not in stop_words]
    return " ".join(words)
reduced_df['clean_text']=reduced_df['content'].apply(clean_text)


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words=1000
max_len=300

token=Tokenizer(num_words=max_words, oov_token="<00V>")
token.fit_on_texts(reduced_df['clean_text'])
sequences=token.texts_to_sequences(reduced_df['clean_text'])
pad=pad_sequences(sequences, maxlen=max_len, padding='post')

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(pad, y_encoded, test_size=0.2, random_state=42)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Embedding, GlobalMaxPooling1D, Dense, Dropout

In [12]:
model=Sequential()

model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.build(input_shape=(None, max_len))
model.summary()

c:\Users\Madhurjya\Desktop\b drive\projectss\fake_news\fakenews\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 128)       │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 296, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 218,369 (853.00 KB)

 Trainable params: 218,369 (853.00 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [14]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, callbacks=[early_stop])

Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7211 - loss: 0.5299 - val_accuracy: 0.9094 - val_loss: 0.1759
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9578 - loss: 0.1546 - val_accuracy: 0.9719 - val_loss: 0.0893
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9777 - loss: 0.0802 - val_accuracy: 0.9734 - val_loss: 0.0712
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9887 - loss: 0.0478 - val_accuracy: 0.9766 - val_loss: 0.0661
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9914 - loss: 0.0358 - val_accuracy: 0.9781 - val_loss: 0.0624
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9926 - loss: 0.0280 - val_accuracy: 0.9812 - val_loss: 0.0598
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9961 - loss: 0.0186 - val_accuracy: 0.9750 - val_loss: 0.0696
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9977 - loss: 0.0084 - val_accuracy: 0.9781 - v

In [15]:
test_loss, test_accuracy=model.evaluate(X_test, y_test)
print("Test Accuracy: ", test_accuracy)

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9850 - loss: 0.0658
Test Accuracy:  0.9850000143051147


In [16]:
# Save CNN model
model.save("cnn.keras")

print("CNN model saved successfully!")

CNN model saved successfully!


In [17]:
import joblib

# Save tokenizer
joblib.dump(token, "cnn_tokenizer.pkl")

print("Tokenizer saved successfully!")

Tokenizer saved successfully!
